In [1]:
# ==================================================================================
# IMPROVED EEG-TO-TEXT MODEL TRAINING SCRIPT
# Key Improvements:
# 1. Fixed loss weight balance (text is now primary objective)
# 2. Improved teacher forcing schedules with curriculum learning
# 3. Added beam search decoding for better generation quality
# 4. Better gradient flow and training stability
# ==================================================================================

import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from scipy.sparse import coo_matrix
import time
import math
import random
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import json
import evaluate as hf_evaluate

# ==================================================================================
# FOCAL LOSS SETUP (BCEWithLogitsLoss for multi-label)
# ==================================================================================
print("Using BCEWithLogitsLoss for multi-label object classification")
focal_loss_criterion = nn.BCEWithLogitsLoss()


/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


Using BCEWithLogitsLoss for multi-label object classification


In [2]:
# ==================================================================================
# CONSTANTS AND CONFIGURATION
# ==================================================================================
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name_qwen.json"

TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 8

NUM_COLORS = 12
NUM_OBJECTS = 90

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Metadata config: {NUM_COLORS} colors, {NUM_OBJECTS} objects")

# ==================================================================================
# IMPROVED LOSS WEIGHTS - TEXT IS PRIMARY OBJECTIVE
# ==================================================================================
TEXT_LOSS_WEIGHT = 1.0       # Primary objective
COLOR_LOSS_WEIGHT = 0.1      # Reduced from 1.0
OBJECT_LOSS_WEIGHT = 0.5     # Reduced from 10.0

print(f"\n=== LOSS WEIGHTS ===")
print(f"Text Loss Weight:   {TEXT_LOSS_WEIGHT} (PRIMARY OBJECTIVE)")
print(f"Color Loss Weight:  {COLOR_LOSS_WEIGHT}")
print(f"Object Loss Weight: {OBJECT_LOSS_WEIGHT}")
print("Rationale: Text generation is the main task. Metadata should be auxiliary.\n")


Using device: cuda
Metadata config: 12 colors, 90 objects

=== LOSS WEIGHTS ===
Text Loss Weight:   1.0 (PRIMARY OBJECTIVE)
Color Loss Weight:  0.1
Object Loss Weight: 0.5
Rationale: Text generation is the main task. Metadata should be auxiliary.



In [3]:
# ==================================================================================
# GRANGER CAUSALITY MATRIX CREATION
# ==================================================================================
def create_granger_causality_matrix(eeg_batch):
    eeg_sample = eeg_batch[0].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j:
                continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            min_len = 20
            if len(ts_i) < min_len or len(ts_j) < min_len:
                causality_matrix[i, j] = 0.0
                continue
            data = np.vstack([ts_j, ts_i]).T
            try:
                current_maxlag = min(5, len(data)//2 - 2)
                if current_maxlag < 1:
                    causality_matrix[i, j] = 0.0
                    continue
                results = grangercausalitytests(data, maxlag=current_maxlag, verbose=False)
                p_value = results[current_maxlag][0]['ssr_ftest'][1]
                if p_value < 0.05:
                    causality_matrix[i, j] = 1.0
            except Exception as e:
                causality_matrix[i, j] = 0.0

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)

    if edge_attr is None:
        edge_attr = torch.tensor([], dtype=torch.float)
    elif edge_attr.ndim == 0:
        edge_attr = edge_attr.unsqueeze(0)

    return edge_index.to(torch.long), edge_attr.to(torch.float)

# ==================================================================================
# DATASET AND DATALOADER
# ==================================================================================
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')

        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype(np.int64))

        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)

    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), meta_batch.float(), text_padded

In [4]:
# ==================================================================================
# MODEL COMPONENTS
# ==================================================================================

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers,
                          bidirectional=True, dropout=dropout if num_layers > 1 else 0,
                          batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, edge_index, edge_attr):
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]

        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1])

        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)

        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))

        temporal_features = x.reshape(batch_size, num_timesteps, -1)
        encoder_outputs, encoder_hidden = self.rnn(temporal_features)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        return encoder_outputs, encoder_hidden

class LuongAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim, dec_dim)

    def forward(self, decoder_hidden, encoder_outputs):
        src_len = encoder_outputs.shape[0]
        attn_energies = self.attn(encoder_outputs)
        scores = torch.bmm(decoder_hidden.permute(1, 0, 2), attn_energies.permute(1, 2, 0))
        attn_weights = F.softmax(scores, dim=2)
        context = torch.bmm(attn_weights, encoder_outputs.permute(1, 0, 2))
        return context, attn_weights.squeeze(1)

class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects, 
                 color_emb_dim=16, object_feature_dim=128):
        super().__init__()
        
        self.color_embedding = nn.Embedding(num_colors, color_emb_dim)
        
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, object_feature_dim)
        )

        self.output_dim = color_emb_dim + object_feature_dim

    def forward(self, metadata):
        color_ids = metadata[:, 0].long()
        object_features_raw = metadata[:, 1:]
        object_features_raw = object_features_raw.float()

        color_vec = self.color_embedding(color_ids)
        object_vec = self.object_processor(object_features_raw)

        combined_features = torch.cat([color_vec, object_vec], dim=1)
        
        return combined_features

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        enc_dim = enc_hidden * 2

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = LuongAttention(enc_dim, dec_hidden)

        self.rnn_input_dim = emb_dim + enc_dim + meta_features_dim + enc_dim
        self.rnn = nn.GRU(self.rnn_input_dim, dec_hidden, num_layers, dropout=dropout if num_layers > 1 else 0)

        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_dim, dec_hidden)

    def init_hidden(self, encoder_hidden):
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last_layer_hidden = hidden[-1]
        encoder_hidden_cat = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        bridged_hidden = torch.tanh(self.bridge(encoder_hidden_cat))
        decoder_initial_hidden = bridged_hidden.unsqueeze(0).repeat(self.num_layers, 1, 1)
        return decoder_initial_hidden

    def forward(self, token, decoder_hidden, encoder_outputs, meta_features, global_eeg_context):
        token = token.unsqueeze(0)
        embedded = self.dropout(self.embedding(token))
        context, attn_weights = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)
        
        meta_features_unsqueezed = meta_features.unsqueeze(0)
        global_eeg_context_unsqueezed = global_eeg_context.unsqueeze(0)
        context_permuted = context.permute(1, 0, 2)

        rnn_input = torch.cat((
            embedded,
            context_permuted,
            meta_features_unsqueezed,
            global_eeg_context_unsqueezed
        ), dim=2)

        output, hidden = self.rnn(rnn_input, decoder_hidden)
        prediction = self.fc_out(output.squeeze(0))

        return prediction, hidden, context.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_objects, enc_hidden=256, dec_hidden=256,
                 pad_id=0, dropout=0.2, color_emb_dim=16, object_feature_dim=128, emb_dim=256, dec_layers=2):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, num_layers=dec_layers)
        
        self.meta_encoder = MetadataEncoder(
            num_colors, 
            num_objects,
            color_emb_dim, 
            object_feature_dim
        )

        meta_features_dim = self.meta_encoder.output_dim
        enc_dim = enc_hidden * 2

        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden,
                                 meta_features_dim, dec_layers, pad_id, dropout)

        self.meta_head = nn.Sequential(
            nn.Linear(enc_dim, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_objects)
        )
        self.num_colors = num_colors
        self.num_objects = num_objects

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr, 
                text_teacher_forcing_ratio=0.5, meta_teacher_forcing_ratio=1.0):
        batch_size = eeg.shape[0]
        target_len = target_text.shape[1]
        target_vocab_size = self.decoder.vocab_size

        encoder_outputs, encoder_hidden = self.encoder(eeg, edge_index, edge_attr)
        decoder_hidden = self.decoder.init_hidden(encoder_hidden)

        hidden_reshaped = encoder_hidden.view(self.encoder.rnn.num_layers, 2, batch_size, -1)
        last_layer_hidden = hidden_reshaped[-1]
        global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

        meta_preds = self.meta_head(global_eeg_context)
        pred_color = meta_preds[:, :self.num_colors]
        pred_object = meta_preds[:, self.num_colors:]

        use_true_meta = random.random() < meta_teacher_forcing_ratio
        
        if use_true_meta:
            meta_features = self.meta_encoder(metadata)
        else:
            with torch.no_grad():
                pred_color_id_vec = pred_color.argmax(dim=-1).float().unsqueeze(1)
                pred_object_vec = (torch.sigmoid(pred_object) > 0.5).float()
                predicted_meta_vector = torch.cat([pred_color_id_vec, pred_object_vec], dim=1)
            
            meta_features = self.meta_encoder(predicted_meta_vector)

        outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(eeg.device)
        decoder_input = target_text[:, 0]

        for t in range(1, target_len):
            output, decoder_hidden, _ = self.decoder(
                decoder_input,
                decoder_hidden,
                encoder_outputs,
                meta_features,
                global_eeg_context
            )

            outputs[t] = output
            teacher_force = random.random() < text_teacher_forcing_ratio
            top1 = output.argmax(1)
            decoder_input = target_text[:, t] if teacher_force else top1

        return outputs[1:].permute(1, 0, 2), pred_color, pred_object

In [5]:
# ==================================================================================
# IMPROVED TEACHER FORCING SCHEDULES
# ==================================================================================
def get_teacher_forcing_ratio(epoch, total_epochs, start=1.0, end=0.3, warmup=10):
    """
    Gradual curriculum learning schedule for text generation.
    
    Args:
        epoch: Current epoch (1-indexed)
        total_epochs: Total training epochs
        start: Initial ratio (1.0 = 100% teacher forcing)
        end: Final ratio (0.3 = 30% teacher forcing)
        warmup: Keep at start ratio for this many epochs
    
    Returns:
        Teacher forcing ratio for this epoch
    """
    if epoch <= warmup:
        return start
    
    progress = (epoch - warmup) / (total_epochs - warmup)
    return start + (end - start) * progress

def get_meta_teacher_forcing_ratio(epoch, total_epochs):
    """
    Conservative schedule for metadata - keeps true metadata longer.
    
    Args:
        epoch: Current epoch (1-indexed)
        total_epochs: Total training epochs
        
    Returns:
        Metadata teacher forcing ratio
    """
    if epoch <= 20:
        return 1.0
    elif epoch <= 30:
        return 1.0 - 0.3 * ((epoch - 20) / 10)
    else:
        return 0.7


In [6]:
# ==================================================================================
# TRAINING FUNCTION
# ==================================================================================
def train_one_epoch(model, loader, optimizer, text_criterion, color_criterion, object_criterion,
                    granger_edge_index, granger_edge_attr, 
                    text_loss_weight, color_loss_weight, object_loss_weight,
                    text_teacher_forcing_ratio=0.5,
                    meta_teacher_forcing_ratio=1.0):
    model.train()
    total_loss = 0.0
    total_loss_t, total_loss_c, total_loss_o = 0.0, 0.0, 0.0
    progress_bar = tqdm(loader, desc="Training", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)

        optimizer.zero_grad()

        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, 
            granger_edge_index, granger_edge_attr, 
            text_teacher_forcing_ratio=text_teacher_forcing_ratio,
            meta_teacher_forcing_ratio=meta_teacher_forcing_ratio
        )

        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), 
                               txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, 0].long())
        loss_o = object_criterion(pred_object, meta_b[:, 1:].float())

        loss = (text_loss_weight * loss_t) + \
               (color_loss_weight * loss_c) + \
               (object_loss_weight * loss_o)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()

        total_loss += loss.item()
        total_loss_t += loss_t.item()
        total_loss_c += loss_c.item()
        total_loss_o += loss_o.item()

        progress_bar.set_postfix(
            loss=loss.item(), 
            txt=loss_t.item(), 
            clr=loss_c.item(), 
            obj=loss_o.item()
        )

    n = len(loader)
    return total_loss / n, total_loss_t / n, total_loss_c / n, total_loss_o / n


In [7]:
# ==================================================================================
# EVALUATION FUNCTION
# ==================================================================================
@torch.no_grad()
def evaluate(model, loader, text_criterion, color_criterion, object_criterion,
             granger_edge_index, granger_edge_attr, 
             text_loss_weight, color_loss_weight, object_loss_weight):
    model.eval()
    total_loss = 0.0
    total_loss_t, total_loss_c, total_loss_o = 0.0, 0.0, 0.0
    progress_bar = tqdm(loader, desc="Evaluating", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)

        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, 
            granger_edge_index, granger_edge_attr, 
            text_teacher_forcing_ratio=0.0,
            meta_teacher_forcing_ratio=1.0
        )

        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), 
                               txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, 0].long())
        loss_o = object_criterion(pred_object, meta_b[:, 1:].float())

        loss = (text_loss_weight * loss_t) + \
               (color_loss_weight * loss_c) + \
               (object_loss_weight * loss_o)
        
        total_loss += loss.item()
        total_loss_t += loss_t.item()
        total_loss_c += loss_c.item()
        total_loss_o += loss_o.item()
        
        progress_bar.set_postfix(
            loss=loss.item(), 
            txt=loss_t.item(), 
            clr=loss_c.item(), 
            obj=loss_o.item()
        )

    n = len(loader)
    return total_loss / n, total_loss_t / n, total_loss_c / n, total_loss_o / n

In [8]:
# ==================================================================================
# BEAM SEARCH DECODER
# ==================================================================================
@torch.no_grad()
def beam_search_decode(model, eeg_signal, edge_index, edge_attr,
                       beam_width=5, max_len=100, length_penalty=0.6):
    """
    Beam search decoder for better quality generation.
    
    Args:
        beam_width: Number of beams (5 is a good default)
        length_penalty: Penalty for shorter sequences (0.6 encourages longer outputs)
    
    Returns:
        best_sequence: List of token IDs
        best_score: Log probability score
    """
    model.eval()
    eeg_signal = eeg_signal.unsqueeze(0).to(device)

    encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)
    
    hidden_reshaped = encoder_hidden.view(model.encoder.rnn.num_layers, 2, 1, -1)
    last_layer_hidden = hidden_reshaped[-1]
    global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

    meta_preds_logits = model.meta_head(global_eeg_context)
    pred_color_logits = meta_preds_logits[:, :model.num_colors]
    pred_object_logits = meta_preds_logits[:, model.num_colors:]

    pred_color_id_vec = pred_color_logits.argmax(dim=-1).float().unsqueeze(1)
    pred_object_vec = (torch.sigmoid(pred_object_logits) > 0.5).float()
    predicted_meta_vector = torch.cat([pred_color_id_vec, pred_object_vec], dim=1)
    predicted_meta_features = model.meta_encoder(predicted_meta_vector)

    decoder_hidden = model.decoder.init_hidden(encoder_hidden)

    beams = [(torch.tensor([SOS_ID], device=device), 0.0, decoder_hidden)]
    completed_beams = []

    for step in range(max_len):
        candidates = []
        
        for seq, score, hidden in beams:
            if seq[-1].item() == EOS_ID:
                completed_beams.append((seq, score))
                continue
            
            input_token = seq[-1].unsqueeze(0)
            prediction, new_hidden, _ = model.decoder(
                input_token, hidden, encoder_outputs,
                predicted_meta_features, global_eeg_context
            )
            
            log_probs = F.log_softmax(prediction.squeeze(0), dim=-1)
            topk_log_probs, topk_ids = torch.topk(log_probs, beam_width)
            
            for i in range(beam_width):
                new_seq = torch.cat([seq, topk_ids[i].unsqueeze(0)])
                new_score = score + topk_log_probs[i].item()
                candidates.append((new_seq, new_score, new_hidden))
        
        if not candidates:
            break
        
        candidates.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
        beams = candidates[:beam_width]
        
        if len(completed_beams) >= beam_width:
            break
    
    completed_beams.extend(beams)
    
    if not completed_beams:
        return torch.tensor([SOS_ID, EOS_ID], device=device), 0.0
    
    completed_beams.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
    best_seq, best_score = completed_beams[0]
    
    return best_seq, best_score

@torch.no_grad()
def generate_end_to_end_with_beam_search(model, eeg_signal, edge_index, edge_attr,
                                         sample_idx, beam_width=5):
    """
    Wrapper that uses beam search and returns formatted output.
    """
    model.eval()
    
    eeg_signal_batched = eeg_signal.unsqueeze(0).to(device)
    encoder_outputs, encoder_hidden = model.encoder(eeg_signal_batched, edge_index, edge_attr)
    hidden_reshaped = encoder_hidden.view(model.encoder.rnn.num_layers, 2, 1, -1)
    last_layer_hidden = hidden_reshaped[-1]
    global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
    
    meta_preds_logits = model.meta_head(global_eeg_context)
    pred_color_logits = meta_preds_logits[:, :model.num_colors]
    pred_object_logits = meta_preds_logits[:, model.num_colors:]
    
    pred_color_for_print = pred_color_logits.argmax(dim=-1).item()
    object_probs = torch.sigmoid(pred_object_logits)
    pred_object_ids_list = (object_probs > 0.5).nonzero(as_tuple=True)[1].tolist()
    
    best_seq, best_score = beam_search_decode(
        model, eeg_signal, edge_index, edge_attr,
        beam_width=beam_width, max_len=100
    )
    
    if best_seq.numel() > 2:
        predicted_text_ids = best_seq[1:-1] if best_seq[-1].item() == EOS_ID else best_seq[1:]
        predicted_text = tokenizer.decode(predicted_text_ids.tolist(), skip_special_tokens=True)
    else:
        predicted_text = ""
    
    return predicted_text, pred_color_for_print, pred_object_ids_list


In [9]:
# ==================================================================================
# MAIN TRAINING LOOP - FIXED SCHEDULING
# ==================================================================================
if __name__ == "__main__":
    # Create dataset and loaders
    dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
    N = len(dataset)
    n_train = int(N * TRAIN_PCT)
    n_val = int(N * VAL_PCT)
    n_test = N - n_train - n_val
    g = torch.Generator().manual_seed(42)
    train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)

    # Create Granger matrix
    print("Creating Granger Causality matrix...")
    try:
        eeg_b, _, _ = next(iter(train_loader))
        granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
        num_channels = eeg_b.shape[1]

        granger_edge_index, granger_edge_attr = add_self_loops(
            granger_edge_index,
            edge_attr=granger_edge_attr,
            num_nodes=num_channels,
            fill_value=1.0
        )

        if granger_edge_attr is None:
            granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float)

        granger_edge_index = granger_edge_index.to(torch.long).to(device)
        granger_edge_attr = granger_edge_attr.to(torch.float32).to(device)
        print(f"Granger matrix created: {granger_edge_index.shape}")

    except Exception as e:
        print(f"Error creating Granger matrix: {e}. Using fallback.")
        num_channels = 62
        edge_index = torch.combinations(torch.arange(num_channels), r=2).t().contiguous()
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        edge_index, _ = add_self_loops(edge_index, num_nodes=num_channels)
        granger_edge_index = edge_index.to(torch.long).to(device)
        granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float32).to(device)

    # UPDATED: Instantiate model with new parameters (no num_categories)
    model = Seq2Seq(
        text_vocab_size=TEXT_VOCAB_SIZE,
        num_colors=NUM_COLORS,
        num_objects=NUM_OBJECTS,
        pad_id=PAD_ID,
        dropout=0.4,  # ← Changed from 0.2 to 0.4 (DOUBLE the dropout)
        enc_hidden=256,
        dec_hidden=256,
        emb_dim=256,
        dec_layers=2
    ).to(device)

    print(f"Model instantiated on '{device}'.")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    # UPDATED: Setup losses (removed category_criterion)
    object_criterion = focal_loss_criterion
    text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)
    color_criterion = nn.CrossEntropyLoss()
    # REMOVED: category_criterion

    optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=0.05)
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.2, patience=2, verbose=True)
    
    EPOCHS = 40
    best_val_loss = float('inf')
    
    # UPDATED WEIGHTS
    TEXT_LOSS_WEIGHT = 1.0
    COLOR_LOSS_WEIGHT = 0.1
    OBJECT_LOSS_WEIGHT = 0.5

    print("\n--- Starting Training ---")
    print(f"Text Loss Weight:   {TEXT_LOSS_WEIGHT} (PRIMARY)")
    print(f"Color Loss Weight:  {COLOR_LOSS_WEIGHT}")
    print(f"Object Loss Weight: {OBJECT_LOSS_WEIGHT}")

    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()

        # UPDATED: Dynamic teacher forcing ratios
        text_tf_ratio = get_teacher_forcing_ratio(epoch, EPOCHS, start=1.0, end=0.3, warmup=10)
        meta_tf_ratio = get_meta_teacher_forcing_ratio(epoch, EPOCHS)
        
        print(f"\n[Epoch {epoch}/{EPOCHS}] Text TF: {text_tf_ratio:.3f} | Meta TF: {meta_tf_ratio:.3f}")

        # UPDATED: Pass all weights and both ratios
        train_loss, tr_t, tr_c, tr_o = train_one_epoch(
            model, train_loader, optimizer,
            text_criterion, color_criterion, object_criterion,
            granger_edge_index, granger_edge_attr, 
            TEXT_LOSS_WEIGHT, COLOR_LOSS_WEIGHT, OBJECT_LOSS_WEIGHT,
            text_teacher_forcing_ratio=text_tf_ratio,
            meta_teacher_forcing_ratio=meta_tf_ratio
        )
        
        val_loss, val_t, val_c, val_o = evaluate(
            model, val_loader,
            text_criterion, color_criterion, object_criterion,
            granger_edge_index, granger_edge_attr, 
            TEXT_LOSS_WEIGHT, COLOR_LOSS_WEIGHT, OBJECT_LOSS_WEIGHT
        )

        scheduler.step(val_loss)
        end_time = time.time()
        epoch_mins = int((end_time - start_time) / 60)
        epoch_secs = int((end_time - start_time) % 60)

        print(f'Epoch: {epoch:02} | Time: {epoch_mins}m {epoch_secs}s')
        print(f'  Train Loss: {train_loss:.4f} | Txt: {tr_t:.4f} | Clr: {tr_c:.4f} | Obj: {tr_o:.4f}')
        print(f'  Val Loss:   {val_loss:.4f} | Txt: {val_t:.4f} | Clr: {val_c:.4f} | Obj: {val_o:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            model_save_path = 'eeg-meta-text-qwen-refactored-model.pt'
            torch.save(model.state_dict(), model_save_path)
            print(f"  ✓ Val loss improved → Saved to '{model_save_path}'")
        else:
            print(f"  ✗ Val loss did not improve")

    print("\n--- Training Complete ---")

Creating Granger Causality matrix...


/home/poorna/venvs/torch/lib64/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


Granger matrix created: torch.Size([2, 2991])
Model instantiated on 'cuda'.
Total parameters: 19,875,552

--- Starting Training ---
Text Loss Weight:   1.0 (PRIMARY)
Color Loss Weight:  0.1
Object Loss Weight: 0.5

[Epoch 1/40] Text TF: 1.000 | Meta TF: 1.000


/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training:   0%|          | 0/2800 [00:00<?, ?it/s]

...

Training:   0%|          | 0/2800 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/350 [00:00<?, ?it/s]

Epoch: 05 | Time: 11m 23s
  Train Loss: 5.0894 | Txt: 4.8166 | Clr: 2.2461 | Obj: 0.0965
  Val Loss:   6.6232 | Txt: 6.3552 | Clr: 2.2079 | Obj: 0.0943
  ✗ Val loss did not improve

[Epoch 6/40] Text TF: 1.000 | Meta TF: 1.000


Training:   0%|          | 0/2800 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [10]:
# ==================================================================================
# UPDATED INFERENCE WITH BEAM SEARCH
# ==================================================================================
# Run inference
NUM_SAMPLES = 20
BEAM_WIDTH = 5  # Increase for better quality (slower)

print(f"\n--- Running Beam Search Inference (beam_width={BEAM_WIDTH}) on {NUM_SAMPLES} Samples ---")

predictions = []
references = []

for i in range(NUM_SAMPLES):
    eeg_sample, meta_sample, true_text_ids = test_ds[i]

    # Extract true metadata
    true_color_id = int(meta_sample[0].item())
    true_object_vector = meta_sample[1:]
    true_object_ids_tensors = true_object_vector.nonzero(as_tuple=True)[0]
    true_object_names = [object_mapping.get(str(id_item), f"ID:{id_item}") 
                         for id_item in true_object_ids_tensors.tolist()]
    if not true_object_names:
        true_object_names = ["None"]

    # UPDATED: Use beam search
    predicted_text, pred_color, pred_object_ids = generate_end_to_end_with_beam_search(
        model, eeg_sample, granger_edge_index, granger_edge_attr,
        sample_idx=i, beam_width=BEAM_WIDTH
    )

    # Decode true text
    true_text_ids_list = true_text_ids.long().tolist()
    true_text = tokenizer.decode(true_text_ids_list, skip_special_tokens=True)

    predictions.append(predicted_text)
    references.append(true_text)

    pred_object_names = [object_mapping.get(str(oid), f"ID:{oid}") 
                         for oid in pred_object_ids]
    if not pred_object_names:
        pred_object_names = ["None"]

    if i < 5:  # Print first 5 samples
        print(f"\n--- Sample {i+1}/{NUM_SAMPLES} ---")
        print(f"TRUE:      {true_text}")
        print(f"PREDICTED: {predicted_text}")
        print(f"Metadata - Color: T={true_color_id} P={pred_color} | Objects: T={','.join(true_object_names[:3])} P={','.join(pred_object_names[:3])}")


--- Running Beam Search Inference (beam_width=5) on 20 Samples ---


NameError: name 'object_mapping' is not defined